# setup + imports

In [3]:
import random
import numpy as np
import pandas as pd
import torch

In [4]:
animals = [l.strip().lower() for l in open('animals_pool.txt') if l.strip()]
questions = [l.strip().lower() for l in open('questions_pool.txt') if l.strip()]
dev = set(pd.read_csv('dev.csv')['animal'].str.strip().str.lower())
test1 = set(pd.read_csv('test1.csv')['animal'].str.strip().str.lower())

In [5]:
cands = []                             
for i in range(len(animals)):
    if animals[i] not in dev and animals[i] not in test1:
        cands.append(i)
print(len(animals), len(questions), len(cands))  

1472 559 822


# table + model

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/ecaterina/practice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
NAME = 'Qwen/Qwen2.5-3B-Instruct'
REV = 'aa8e72537993ba99e69dfaafa59ed015b17504d1'
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

tok = AutoTokenizer.from_pretrained(NAME, revision=REV)
tok.padding_side = 'left'               # so the last position is the real end of every prompt
model = AutoModelForCausalLM.from_pretrained(NAME, revision=REV, torch_dtype=torch.float16).to(DEVICE).eval()

PROMPT = ("You are answering a question about one specific animal.\n"
          "The animal is: {animal}.\n"
          "Answer with a single word, yes or no.\n"
          "Question: {question}")

def chat_text(animal, question):
    messages = [{'role': 'user', 'content': PROMPT.format(animal=animal, question=question)}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


Loading weights: 100%|███████████████████████| 434/434 [00:10<00:00, 40.88it/s]


In [13]:
yesids = [tok.encode('Yes', add_special_tokens=False)[0], tok.encode('yes', add_special_tokens=False)[0]]
noids = [tok.encode('No', add_special_tokens=False)[0], tok.encode('no', add_special_tokens=False)[0]]

In [17]:
table = np.zeros((len(cands), len(questions)), dtype=np.int8)
margin = np.zeros((len(cands), len(questions)), dtype=np.float32)
bs = 32

for i in range(len(cands)):
    texts = [chat_text(animals[cands[i]], q) for q in questions]
    for start in range(0, len(texts), bs):
        enc = tok(texts[start:start+bs], return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits[:, -1, :]
        y = logits[:, yesids].max(dim=1).values
        n = logits[:, noids].max(dim=1).values
        table[i, start:start + bs] = (y > n).cpu().numpy()
        margin[i, start:start + bs] = (y - n).abs().float().cpu().numpy()
    if i % 20 == 0:
        np.save('table.npy', table)
        np.save('margin.npy', margin)
        print(i, '/', len(cands))
np.save('table.npy', table)
np.save('margin.npy', margin)

KeyboardInterrupt: 

In [ ]:
cnt = 0
nodes = {}                    

def newid():
    global cnt
    i = cnt
    cnt = cnt + 1
    return i

def guess_chain(node, group):
    now = node
    for k in range(len(group)):
        if k == len(group) - 1:
            nxt = None                          
        else:
            nxt = newid()
        nodes[now] = {'type': 'guess', 'animal': cands[group[k]], 'wrong': nxt}
        now = nxt

def build(group):
    node = newid()

    if len(group) <= 3:
        guess_chain(node, group)
        return node

    bestq = 0
    bestscore = -1
    for q in range(len(questions)):
        nryes = 0
        for a in group:
            if table[a][q] == 1:
                nryes += 1
        nrno = len(group) - nryes
        score = min(nryes, nrno)
        if score > bestscore:
            bestscore = score
            bestq = q

    if bestscore == 0:
        guess_chain(node, group)
        return node

    yes = [a for a in group if table[a][bestq] == 1]
    no = [a for a in group if table[a][bestq] == 0]
    yesnode = build(yes)
    nonode = build(no)
    nodes[node] = {'type': 'ask', 'question': bestq, 'yes': yesnode, 'no': nonode}
    return node

table = np.load('table.npy')
build(list(range(len(cands))))
print(len(nodes))

# submission

In [ ]:
import json, zipfile

node_list = []
for i in range(len(nodes)):
    node_list.append(nodes[i])

policy = {
    'schema': 'ioai-2026-animal-policy-v1',
    'animals_sha256': '87eb93cc5d2a238a38daaeb201c394db935153cc17b03a44a7642f0db9ff35dc',
    'questions_sha256': 'd2bd060866382f3dd1329112e73fbd6bf2397352d02469434b7b3d199bd2ca62',
    'root': 0,
    'nodes': node_list,
}
with open('policy.json', 'w') as f:
    json.dump(policy, f)
with zipfile.ZipFile('submission.zip', 'w', compression=zipfile.ZIP_STORED) as z:
    z.write('policy.json')

In [ ]:
!python3 validate_submission.py submission.zip


KeyboardInterrupt



mps
mps:0
False
True
